# Google AlloyDB for PostgreSQL - `AlloyDBToolSpec`

> [AlloyDB](https://cloud.google.com/alloydb) is a fully managed relational database service that offers high performance, seamless integration, and impressive scalability. AlloyDB is 100% compatible with PostgreSQL. Extend your database application to build AI-powered experiences leveraging AlloyDB's LlamaIndex integrations.

This notebook goes over how to use [AlloyDB AI functions](https://cloud.google.com/alloydb/docs/ai/analyze-sentiment) as LlamaIndex agent tools with the `AlloyDBToolSpec` class. It analyzes sentiment, summarizes text and evaluates natural-language conditions inside the database.

Learn more about the package on [GitHub](https://github.com/googleapis/llama-index-alloydb-pg-python/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googleapis/llama-index-alloydb-pg-python/blob/main/samples/llama_index_ai_tools.ipynb)

## Before you begin

To run this notebook, you will need to do the following:

 * [Create a Google Cloud Project](https://developers.google.com/workspace/guides/create-project)
 * [Enable the AlloyDB API](https://console.cloud.google.com/flows/enableapi?apiid=alloydb.googleapis.com)
 * [Create a AlloyDB cluster and instance](https://cloud.google.com/alloydb/docs/cluster-create) running PostgreSQL 17 or later.
 * [Create a AlloyDB database.](https://cloud.google.com/alloydb/docs/quickstart/create-and-connect)
 * [Add a User to the database.](https://cloud.google.com/alloydb/docs/database-users/about)
 * [Set up AlloyDB AI](https://cloud.google.com/alloydb/docs/ai/analyze-sentiment#before-you-begin): install the `google_ml_integration` extension and turn on the `google_ml_integration.enable_ai_query_engine` and `google_ml_integration.enable_preview_ai_functions` flags.

### 🦙 Library Installation
Install the integration library, `llama-index-alloydb-pg`.

In [ ]:
%pip install --upgrade --quiet llama-index-alloydb-pg llama-index-llms-google-genai

**Colab only:** Uncomment the following cell to restart the kernel or use the button to restart the kernel. For Vertex AI Workbench you can restart the terminal using the button on top.

In [ ]:
# # Automatically restart kernel after installs so that your environment can access the new packages
# import IPython

# app = IPython.Application.instance()
# app.kernel.do_shutdown(True)

### 🔐 Authentication
Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

* If you are using Colab to run this notebook, use the cell below and continue.
* If you are using Vertex AI Workbench, check out the setup instructions [here](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env).

In [ ]:
from google.colab import auth

auth.authenticate_user()

### ☁ Set Your Google Cloud Project
Set your Google Cloud project so that you can leverage Google Cloud resources within this notebook.

If you don't know your project ID, try the following:

* Run `gcloud config list`.
* Run `gcloud projects list`.
* See the support page: [Locate the project ID](https://support.google.com/googleapi/answer/7014113).

In [ ]:
# @markdown Please fill in the value below with your Google Cloud project ID and then run the cell.

PROJECT_ID = "my-project-id"  # @param {type:"string"}

# Set the project id
!gcloud config set project {PROJECT_ID}

## Basic Usage

### Set AlloyDB database values
Find your database values, in the [AlloyDB Instances page](https://console.cloud.google.com/alloydb/clusters).

In [ ]:
# @title Set Your Values Here { display-mode: "form" }
REGION = "us-central1"  # @param {type: "string"}
CLUSTER = "my-cluster"  # @param {type: "string"}
INSTANCE = "my-primary"  # @param {type: "string"}
DATABASE = "my-database"  # @param {type: "string"}
USER = "postgres"  # @param {type: "string"}
PASSWORD = "my-password"  # @param {type: "string"}

### AlloyDBEngine Connection Pool

`AlloyDBToolSpec` runs the AI functions through an `AlloyDBEngine`, which configures a connection pool to your AlloyDB database. See the [`AlloyDBVectorStore` notebook](./llama_index_vector_store.ipynb) for all the connection options.

In [ ]:
from llama_index_alloydb_pg import AlloyDBEngine

engine = await AlloyDBEngine.afrom_instance(
    project_id=PROJECT_ID,
    region=REGION,
    cluster=CLUSTER,
    instance=INSTANCE,
    database=DATABASE,
    user=USER,
    password=PASSWORD,
)

### Create AlloyDBToolSpec

By default, the AI functions use the model named by the database's `google_ml_integration.default_llm_model` flag. Pass `model_id` to use another registered model, for example `AlloyDBToolSpec(engine, model_id="gemini-2.5-flash")`.

In [ ]:
from llama_index_alloydb_pg import AlloyDBToolSpec

tool_spec = AlloyDBToolSpec(engine)

### Call the AI functions

Each function has a sync and an async version. This notebook uses the async ones.

In [ ]:
await tool_spec.aanalyze_sentiment(
    "I love this product. It works perfectly!"
)  # 'positive'

In [ ]:
article = (
    "AlloyDB for PostgreSQL is a fully managed, PostgreSQL-compatible database service. "
    "Its columnar engine speeds up analytical queries, and AlloyDB AI lets applications "
    "call models such as Gemini directly from SQL."
)
await tool_spec.asummarize(article, instructions="Use one short sentence.")

In [ ]:
await tool_spec.aevaluate_condition("Is Paris the capital of France?")  # True

### Use the tools in an agent

`to_tool_list()` turns the spec into one `FunctionTool` per AI function. The docstrings become the tool descriptions the agent reads.

In [ ]:
tools = tool_spec.to_tool_list()
for tool in tools:
    print(tool.metadata.name)

Give the tools to any LlamaIndex agent, for example a `FunctionAgent` with Gemini. Set `GOOGLE_API_KEY` or configure Vertex AI for the `GoogleGenAI` LLM first.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.google_genai import GoogleGenAI

agent = FunctionAgent(tools=tools, llm=GoogleGenAI(model="gemini-2.5-flash"))
response = await agent.run(
    "Is this review positive or negative? 'The battery died after two days.'"
)
print(response)